In [90]:
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [91]:
dirpath = os.getcwd()
features_path = r"data\gmfeature_table.csv"
data_path = r"C:\Users\marie\rep_codes\udder_project\udder_analysis\long_format_df"
visit_path = r"C:\Users\marie\rep_codes\udder_project\delpro_vms\data\milk_videos_visit.csv"
plot_dir = os.path.join(os.path.normpath(dirpath + os.sep + os.pardir),r"adsa\examples")

In [92]:
# only keep inference cows
file_path = r"C:\Users\marie\rep_codes\udder_project\udder_video\filelist_topred.txt"
with open(file_path, "r") as f:
    files = f.read().split("\n")
inf_cows = np.unique([int(file.split(",")[1].split("_")[0]) for file in files])
inf_cows_df = pd.DataFrame(inf_cows, columns = ["cow"])

In [93]:
df = pd.read_csv(os.path.join(data_path, "lactation_features.csv"))
vdf = pd.read_csv(visit_path)
vdf_selected = vdf[['cow', 'days_in_milk']]
df_merged = df.merge(vdf_selected, on = 'cow')
df_merged = inf_cows_df.merge(df_merged, on = "cow", how = "inner")
len(np.unique(df_merged.cow))

93

In [94]:
# add min teat length, max teat length, min eu distance, max eu distance, min gd distance, max gd distance
df_merged["min_teat"] = [np.nanmin(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["max_teat"]= [np.nanmax(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["min_eu"] = [np.nanmin(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_eu"] = [np.nanmax(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_gd"] = [np.nanmax(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]

C:\Users\marie\AppData\Local\Temp\ipykernel_11336\1974453648.py:6: RuntimeWarning: All-NaN slice encountered
  df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_11336\1974453648.py:7: RuntimeWarning: All-NaN slice encountered
  df_merged["max_gd"] = [np.nanmax(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]


In [95]:
udder_features = ['vol_udder', 'sarea_udder', 'peri_udder', 'area_udder', 'circ_udder', 'exc_udder','min_teat', 'max_teat', 'min_eu', 'max_eu', 'min_gd', 'max_gd']
prod_vars = ['yield_visit_mean', 'interval_sec_mean', 'kickoff_any_perc', 'days_in_milk', "lactation"]

In [96]:
udder_pearson_df = pd.DataFrame(index = udder_features, columns = prod_vars)
udder_pvals_df = pd.DataFrame(index = udder_features, columns = prod_vars)

In [97]:
for u in udder_features:
    for v in prod_vars:
        selected = df_merged[[v, u]].dropna(axis=0) 
        res = stats.pearsonr(selected[u], selected[v])
        udder_pearson_df.loc[u, v] = res.statistic
        udder_pvals_df.loc[u, v] = res.pvalue

In [98]:
udder_pearson_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.533182,0.058121,-0.115881,0.119131,0.406228
sarea_udder,0.546343,-0.012212,-0.175355,0.001151,0.49381
peri_udder,0.602439,-0.050277,-0.293532,0.07588,0.647666
area_udder,0.579538,-0.022086,-0.26476,0.047567,0.636346
circ_udder,-0.304601,0.016505,-0.010242,-0.085916,-0.296907
exc_udder,-0.036396,-0.034738,0.182337,-0.202872,0.016429
min_teat,0.294393,0.162024,-0.011354,-0.002463,0.244553
max_teat,0.145684,0.048539,0.022455,-0.015644,0.104921
min_eu,0.138629,0.174494,0.102083,-0.032132,0.09361
max_eu,0.473633,0.009808,-0.161119,0.176594,0.488335


In [99]:
udder_pvals_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.0,0.592839,0.285145,0.271754,0.000095
sarea_udder,0.0,0.909049,0.098302,0.991408,0.000001
peri_udder,0.0,0.632224,0.004295,0.469737,0.0
area_udder,0.0,0.832659,0.009915,0.648919,0.0
circ_udder,0.005396,0.882997,0.927235,0.442799,0.006755
exc_udder,0.739364,0.750842,0.092899,0.061019,0.88066
min_teat,0.003975,0.118716,0.913511,0.981207,0.017525
max_teat,0.1612,0.64223,0.829907,0.881037,0.314213
min_eu,0.182691,0.092554,0.327556,0.758504,0.369501
max_eu,0.000001,0.925249,0.120816,0.088635,0.000001


In [100]:
udder_pvals_adj_df = udder_pvals_df
col_num = udder_pvals_adj_df.shape[1]
for col in range(col_num):
    ps = udder_pvals_adj_df.iloc[:, col].to_list()
    udder_pvals_adj_df.iloc[:, col] = stats.false_discovery_control(ps, method='bh') 

In [101]:
udder_pvals_adj_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.0,0.925249,0.524071,0.815262,0.000189
sarea_udder,0.0,0.925249,0.289958,0.991408,0.000002
peri_udder,0.0,0.925249,0.051543,0.939473,0.0
area_udder,0.0,0.925249,0.059489,0.991408,0.0
circ_udder,0.008094,0.925249,0.927235,0.939473,0.011579
exc_udder,0.739364,0.925249,0.289958,0.354541,0.88066
min_teat,0.006814,0.712294,0.927235,0.991408,0.026288
max_teat,0.19344,0.925249,0.927235,0.991408,0.377056
min_eu,0.199299,0.712294,0.524071,0.991408,0.403092
max_eu,0.000003,0.925249,0.289958,0.354541,0.000002


In [102]:
udder_pvals_df.to_csv(os.path.join(dirpath, "tables", "udder_pvals_df.csv"), index = True)
udder_pearson_df.to_csv(os.path.join(dirpath, "tables", "udder_pearson_df.csv"), index = True)
udder_pvals_adj_df.to_csv(os.path.join(dirpath, "tables", "udder_pvals_adj_df.csv"), index = True)